# Cuaderno de Integracion entre Visual y PLN

In [46]:
from azure.ai.textanalytics import TextAnalyticsClient
from azure.core.credentials import AzureKeyCredential

## Pruebas inciales para vision de Azure, implementacion de reconocimiento de imagenes

```Python
# Instalacion necesaria previa
pip install azure-cognitiveservices-vision-computervision
pip install azure-ai-textanalytics

In [47]:
from azure.cognitiveservices.vision.computervision import ComputerVisionClient
from azure.cognitiveservices.vision.computervision.models import VisualFeatureTypes
from msrest.authentication import CognitiveServicesCredentials

from dotenv import load_dotenv
import os


load_dotenv("credentials.env")

key_VI = os.getenv("VISION_KEY")
endpoint_VI = os.getenv("VISION_ENDPOINT")

client_VI = ComputerVisionClient(endpoint=endpoint_VI, credentials=CognitiveServicesCredentials(key_VI))

In [48]:
url = "https://upload.wikimedia.org/wikipedia/commons/3/3a/Cat03.jpg"

image_analysis = client_VI.analyze_image(url,visual_features=[VisualFeatureTypes.tags], language="es")

for tag in image_analysis.tags:
    print(tag)

{'additional_properties': {}, 'name': 'animal', 'confidence': 0.9991010427474976, 'hint': None}
{'additional_properties': {}, 'name': 'gato', 'confidence': 0.9985039830207825, 'hint': None}
{'additional_properties': {}, 'name': 'mamíferos', 'confidence': 0.9980359077453613, 'hint': None}
{'additional_properties': {}, 'name': 'bigotes', 'confidence': 0.9781772494316101, 'hint': None}
{'additional_properties': {}, 'name': 'gatos de tamaño pequeño a mediano', 'confidence': 0.9777382612228394, 'hint': None}
{'additional_properties': {}, 'name': 'gato doméstico', 'confidence': 0.9691081047058105, 'hint': None}
{'additional_properties': {}, 'name': 'félidos', 'confidence': 0.9345883131027222, 'hint': None}
{'additional_properties': {}, 'name': 'mau árabe', 'confidence': 0.8746515512466431, 'hint': None}
{'additional_properties': {}, 'name': 'gato asiático', 'confidence': 0.8502976894378662, 'hint': None}
{'additional_properties': {}, 'name': 'naranja', 'confidence': 0.769542396068573, 'hint'

In [49]:
models = client_VI.list_models()

for x in models.models_property:
    print(x)

{'additional_properties': {}, 'name': 'landmarks', 'categories': ['outdoor_', '户外_', '屋外_', 'aoarlivre_', 'alairelibre_', 'building_', '建筑_', '建物_', 'edifício_']}


In [50]:
domain = "landmarks"
url = "https://images.pexels.com/photos/338515/pexels-photo-338515.jpeg"
language = "es"

analysis = client_VI.analyze_image_by_domain(domain, url, language)

for landmark in analysis.result["landmarks"]:
    print(landmark["name"])
    print(landmark["confidence"])

Torre Eiffel
0.971265435218811


In [51]:
url = "https://www.vinccihoteles.com/media/uploads/cms_apps/imagenes/la-pedrera-barcelona.jpg?q=pr:sharp/rs:fill/w:900/h:500/g:ce/f:jpg"
language = "es"
max_descriptions = 3

analysis = client_VI.describe_image(url, max_descriptions, language)

for caption in analysis.captions:
    print(caption.text)
    print(caption.confidence)

ciudad con edificios altos con Casa Milà de fondo
0.7334685667683457
Casa Milà de fondo
0.7048118316819237
vista de Casa Milà
0.6901421583158363


### Pruebas de analisis por dominio y descripción de la imágen
Con la url https://www.vinccihoteles.com/media/uploads/cms_apps/imagenes/la-pedrera-barcelona.jpg?q=pr:sharp/rs:fill/w:900/h:500/g:ce/f:jpg he detectado las siguientes descripciones:

* {'text': 'a large building with a clock tower', 'confidence': 0.971, 'hint': None}
* {'text': 'a large building with a clock tower and a street in front of it', 'confidence': 0.971, 'hint': None}
* {'text': 'a large building with a clock tower and a street in front of it', 'confidence': 0.971, 'hint': None}

In [60]:
# import models
from azure.cognitiveservices.vision.computervision.models import OperationStatusCodes

url = "https://github.com/Azure-Samples/cognitive-services-python-sdk-samples/raw/master/samples/vision/images/make_things_happen.jpg"
raw = True
numberOfCharsInOperationId = 36

# SDK call
rawHttpResponse = client_VI.read(url, language="es", raw=True)

# Get ID from returned headers
operationLocation = rawHttpResponse.headers["Operation-Location"]
idLocation = len(operationLocation) - numberOfCharsInOperationId
operationId = operationLocation[idLocation:]

# SDK call
result = client_VI.get_read_result(operationId)

# Get data
if result.status == OperationStatusCodes.succeeded:

    for line in result.analyze_result.read_results[0].lines:
        print(line.text)
        print(line.bounding_box)

print(result)

{'additional_properties': {}, 'status': <OperationStatusCodes.running: 'running'>, 'created_date_time': '2026-03-18T18:56:24Z', 'last_updated_date_time': '2026-03-18T18:56:24Z', 'analyze_result': None}


## Pruebas PLN

In [53]:
from dotenv import load_dotenv
import os

from azure.ai.textanalytics import TextAnalyticsClient
from azure.core.credentials import AzureKeyCredential

# 1️⃣ Cargar .env
load_dotenv("credentials.env")

# 2️⃣ Obtener variables
endpoint_PLN = os.getenv("PLN_ENDPOINT")
key_PLN = os.getenv("PLN_KEY")

# 3️⃣ Crear cliente
client_PLN = TextAnalyticsClient(
    endpoint=endpoint_PLN,
    credential=AzureKeyCredential(key_PLN)
)

In [54]:
documents = [
    "Me encanta aprender el lenguaje de programación Python y trabajar con Azure!",
    "jfevufuwfk"
]

In [55]:
response = client_PLN.analyze_sentiment(documents=documents)

In [56]:
for idx, doc in enumerate(response):
    print(f"Documento {idx+1}:")
    print(f"  Texto: {documents[idx]}")
    print(f"  Sentimiento: {doc.sentiment}")
    print(f"  Confianza: {doc.confidence_scores}")

Documento 1:
  Texto: Me encanta aprender el lenguaje de programación Python y trabajar con Azure!
  Sentimiento: positive
  Confianza: {'positive': 1.0, 'neutral': 0.0, 'negative': 0.0}
Documento 2:
  Texto: jfevufuwfk
  Sentimiento: negative
  Confianza: {'positive': 0.05, 'neutral': 0.43, 'negative': 0.52}


In [57]:
help(client_PLN)
dir(client_PLN)

Help on TextAnalyticsClient in module azure.ai.textanalytics._text_analytics_client object:

class TextAnalyticsClient(azure.ai.textanalytics._base_client.TextAnalyticsClientBase)
 |  TextAnalyticsClient(
 |      endpoint: str,
 |      credential: Union[azure.core.credentials.AzureKeyCredential, azure.core.credentials.TokenCredential],
 |      *,
 |      default_language: Optional[str] = None,
 |      default_country_hint: Optional[str] = None,
 |      api_version: Union[str, azure.ai.textanalytics._base_client.TextAnalyticsApiVersion, NoneType] = None,
 |      **kwargs: Any
 |  ) -> None
 |
 |  The Language service API is a suite of natural language processing (NLP) skills built with the best-in-class
 |  Microsoft machine learning algorithms. The API can be used to analyze unstructured text for
 |  tasks such as sentiment analysis, key phrase extraction, entities recognition,
 |  and language detection, and more.
 |
 |  Further documentation can be found in
 |  https://learn.microsof

['__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__enter__',
 '__eq__',
 '__exit__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_analyze_result_callback',
 '_api_version',
 '_client',
 '_default_country_hint',
 '_default_language',
 '_healthcare_result_callback',
 '_string_index_type_default',
 'analyze_sentiment',
 'begin_abstract_summary',
 'begin_analyze_actions',
 'begin_analyze_healthcare_entities',
 'begin_extract_summary',
 'begin_multi_label_classify',
 'begin_recognize_custom_entities',
 'begin_single_label_classify',
 'close',
 'detect_language',
 'extract_key_phrases',
 'recognize_entities',
 'recognize_linked_entities',
 'recognize_pii_entities']

In [58]:
response2 = client_PLN.detect_language(documents=documents)



for idx, doc in enumerate(response2):
    if not doc.is_error:
        print(f"Documento {idx+1}:")
        print(f"  Texto: {documents[idx]}")
        print(f"  Idioma detectado: {doc.primary_language.name}")
        print(f"  Código: {doc.primary_language.iso6391_name}")
        print(f"  Confianza: {doc.primary_language.confidence_score}")
        print(f"  Estadísticas: {doc.statistics}")
    else:
        print(f"Documento {idx+1} tuvo un error: {doc.error}")

Documento 1:
  Texto: Me encanta aprender el lenguaje de programación Python y trabajar con Azure!
  Idioma detectado: Spanish
  Código: es
  Confianza: 1.0
  Estadísticas: None
Documento 2:
  Texto: jfevufuwfk
  Idioma detectado: English
  Código: en
  Confianza: 0.89
  Estadísticas: None


In [59]:
response3 = client_PLN.recognize_entities(documents=documents)

for idx, doc in enumerate(response3):
    print(f"Documento {idx+1}: {documents[idx]}")
    if not doc.is_error:
        if doc.entities:
            for entity in doc.entities:
                print(f"  Entidad: {entity.text}")
                print(f"    Tipo: {entity.category}")
                print(f"    Subtipo: {entity.subcategory}")
                print(f"    Confianza: {entity.confidence_score}")
        else:
            print("  No se encontraron entidades.")
    else:
        print(f"  Error: {doc.error}")

Documento 1: Me encanta aprender el lenguaje de programación Python y trabajar con Azure!
  Entidad: programación
    Tipo: Skill
    Subtipo: None
    Confianza: 0.66
  Entidad: Python
    Tipo: Skill
    Subtipo: None
    Confianza: 1.0
  Entidad: Azure
    Tipo: Product
    Subtipo: ComputingProduct
    Confianza: 1.0
Documento 2: jfevufuwfk
  No se encontraron entidades.
